<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            <div>We used a mix of custom Parallelization and Evaluator workflows patterns</div>
            <span>How about using Prompt chaining so that we ask the first model and then feed the answer to the second model to judge. If it satisfactory, then we are done otherwise it will provide a better answer and so on. We will feed the answer to the third model, if it okay we are all set, otherwise, it will generate a better answer.</span>
            </span>
        </td>
    </tr>
</table>

### Exercise Solution: Prompt Chaining + Evaluator Optimizer Cascade

We start with a smaller/faster model to draft an initial answer, and then chain through progressively stronger models. Each subsequent model acts as a **gate/evaluator** and **optimizer**: if the current answer is satisfactory, we stop early; otherwise, it provides an improved answer and passes it along the chain.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pydantic import BaseModel

load_dotenv(override=True)

In [ ]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


In [ ]:
# OpenAI Compatible URLs

ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"


In [ ]:
openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url=ANTHROPIC_BASE_URL)
gemini = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:

# Define the structured output schema using a Pydantic class
class AnswerReview(BaseModel):
    satisfactory: bool
    critique: str
    improved_answer: str

# Generate a challenging question for our cascade
question_prompt = """Please come up with a challenging, nuanced question with a succinct answer 
    that I can ask a chain of LLMs to evaluate and refine their intelligence. 
    Include in your question that the answer must be short (two sentences or fewer). 
    Answer only with the question, no explanation."""

q_response = anthropic.chat.completions.create(
    model="claude-haiku-5-5",
    messages=[{"role": "user", "content": question_prompt}],
    reasoning_effort="low"
)
cascade_question = q_response.choices[0].message.content
display(Markdown(f"### Question (generated by `claude-haiku-5-5`)\n> {cascade_question}"))

# List of models from lightweight to stronger
cascade = [
    (ollama, "gpt-oss:20b"),
    (openai, "gpt-6-luna"),
    (gemini, "gemini-4-hydrogen"),
    (anthropic, "claude-sonnet-5-5"),
]

# Step 1: Ask the first model in the chain for an initial answer
first_client, first_model = cascade[0]
response = first_client.chat.completions.create(
    model=first_model,
    messages=[{"role": "user", "content": cascade_question}]
)
current_answer = response.choices[0].message.content
current_author = first_model

display(Markdown(f"### Initial Answer from `{first_model}`\n\n{current_answer}"))

# Step 2: Chain through the remaining models to evaluate and optimize if needed
system_prompt = (
    """You are a strict expert evaluator and editor. Evaluate the provided answer 
    for depth of insight, clarity, and conciseness (two sentences or fewer). 
    If it is already strong and satisfactory, set satisfactory to true and leave 
    improved_answer empty. Otherwise, set satisfactory to false and provide a 
    sharper, better answer in improved_answer."""
)

for client, model_name in cascade[1:]:
    user_prompt = (
        f"Question:\n{cascade_question}\n\n"
        f"Current Answer (from {current_author}):\n{current_answer}"
    )

    print(f"Asking {model_name}")
    
    response = client.beta.chat.completions.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=AnswerReview,
        reasoning_effort="low"
    )

    review = response.choices[0].message.parsed
    
    if review.satisfactory:
        display(Markdown(f"### Approved by `{model_name}`\n**Critique:** {review.critique}\n\nStopping early — answer is satisfactory!"))
        break
    else:
        current_answer = review.improved_answer or current_answer
        current_author = model_name
        display(Markdown(
            f"### Refined by `{model_name}`\n"
            f"**Critique of previous answer:** {review.critique}\n\n"
            f"**New Answer:**\n{current_answer}"
        ))

display(Markdown(f"---\n## Final Answer (by `{current_author}`)\n\n{current_answer}"))